# Unsloth: 2x Faster Fine-Tuning with 70% Less Memory

## What is Unsloth?

Unsloth is a fine-tuning library that rewrites critical transformer kernels (attention, RoPE, cross-entropy) 
in Triton, achieving **2x faster training** and **70% less VRAM** vs standard HuggingFace + bitsandbytes.

It's the most popular fine-tuning acceleration library in 2024-2025, used in the majority of community 
fine-tuning runs for Llama, Qwen, Gemma, and Mistral.

**What Unsloth optimizes:**
- Custom Triton kernels for RoPE (rotary position embeddings)
- Fused attention + causal masking
- Lossless quantization (their 4-bit is more accurate than bitsandbytes NF4)
- 4x faster inference with their custom GGUF export

**The API is a drop-in replacement** for `AutoModelForCausalLM.from_pretrained`:
```python
# Before (standard)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", ...)

# After (Unsloth — same result, 2x faster, 70% less VRAM)  
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("Qwen/Qwen3-0.6B", ...)
```

In [ ]:
# IMPORTANT: Run this cell first, then restart the runtime (Runtime → Restart session)
# Unsloth requires a specific TRL version — installing in fresh session avoids conflicts
!pip install unsloth -q 2>&1 | tail -2
# Verify compatible versions
import unsloth, trl
print(f"Unsloth {unsloth.__version__} | TRL {trl.__version__}")
print("If you see ImportError, restart the runtime and re-run this cell first")

In [ ]:
from unsloth import FastLanguageModel
import torch

# Unsloth's FastLanguageModel — drop-in for AutoModelForCausalLM
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-0.6B",
    max_seq_length=512,
    dtype=None,          # auto-detect (float16 on T4, bfloat16 on A100)
    load_in_4bit=True,   # Unsloth's custom 4-bit (more accurate than bitsandbytes)
)

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Apply LoRA with Unsloth's optimized adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,     # Unsloth recommends 0 dropout for speed
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

# Instruction tuning dataset
alpaca_data = [
    {"instruction": "What is LoRA fine-tuning?",
     "input": "",
     "output": "LoRA (Low-Rank Adaptation) fine-tunes LLMs by adding small trainable rank-decomposition matrices to frozen weights. This reduces trainable parameters by 99% while maintaining quality. It's the most widely used PEFT method."},
    {"instruction": "Explain the transformer architecture.",
     "input": "",
     "output": "Transformers use stacked self-attention layers to process sequences in parallel. Each layer has multi-head attention (attending to different positions) and a feed-forward network. Layer normalization and residual connections ensure stable gradients."},
    {"instruction": "What is RLHF?",
     "input": "",
     "output": "RLHF (Reinforcement Learning from Human Feedback) aligns LLMs with human preferences by: (1) training a reward model on human rankings, (2) fine-tuning the policy with PPO to maximize reward while staying close to the reference model via KL penalty."},
    {"instruction": "How does quantization work?",
     "input": "",
     "output": "Quantization reduces weight precision (e.g., FP16 → INT4), shrinking model size 4× with minimal quality loss. Methods: GPTQ (Hessian-aware rounding), AWQ (activation-aware channel scaling), GGUF (CPU-friendly block quantization)."},
] * 8

# Alpaca format template
ALPACA_PROMPT = """Below is an instruction. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_alpaca(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        texts.append(ALPACA_PROMPT.format(inst, inp, out) + tokenizer.eos_token)
    return {"text": texts}

dataset = Dataset.from_list(alpaca_data).map(format_alpaca, batched=True, remove_columns=["instruction","input","output"])
print(f"Dataset: {len(dataset)} formatted examples")
print(f"Sample:\n{dataset[0]['text'][:300]}...")

In [ ]:
# Train with SFTTrainer — Unsloth patches it automatically
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        output_dir="/tmp/unsloth-out",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=4,
        save_strategy="no",
        report_to="none",
        warmup_steps=5,
        lr_scheduler_type="cosine",
    ),
)

import time
t0 = time.time()
result = trainer.train()
elapsed = time.time() - t0

print(f"\n✅ Unsloth training complete!")
print(f"   Loss: {result.training_loss:.4f}")
print(f"   Runtime: {elapsed:.1f}s")
print(f"   Steps: {result.global_step}")
print(f"   Memory peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# Test generation
FastLanguageModel.for_inference(model)  # enable Unsloth's fast inference mode
inputs = tokenizer([ALPACA_PROMPT.format("What is DPO?", "", "")], return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
response = tokenizer.decode(out[0], skip_special_tokens=True)
print(f"\nGenerated response:\n{response[response.find('Response:')+9:].strip()}")

## Unsloth Key Features

| Feature | Standard QLoRA | Unsloth |
|---------|---------------|---------|
| Speed | 1x baseline | 2x faster |
| VRAM | 1x baseline | 70% less |
| Quality | Same | Same (lossless) |
| Max context | Limited | 4x longer with same VRAM |
| GRPO support | Yes | Yes (optimized) |
| Export | PEFT adapter | GGUF, merged, PEFT |

**When Unsloth is most impactful:**
- Long sequences (>1024 tokens) — RoPE kernel dominates
- 4-bit training — their quantization is tighter than bitsandbytes
- GRPO — 80% less VRAM claim for reasoning model training

**Limitations:**
- Linux/CUDA only (no Windows, no Apple Silicon)
- Some models not yet supported (check unsloth.ai for list)
- Custom kernels can be harder to debug than PyTorch ops